# 🎭 Qwen2.5-1.5B QLoRA — Poetry Fine-Tuning

**Fine-tune Qwen2.5-1.5B on Vietnamese poetry with LoRA + 4-bit quantization.**

Same control tokens (335 special tokens) as PoetryDuelGPT.  
Compares directly against our 31M custom model.

| Step | Time |
|------|------|
| Clone + Install | ~2 min |
| Preprocess + tokenize | ~3 min |
| Stage 1: All genres (5K steps) | ~3 hr T4 |
| Stage 2: Lục Bát fine-tune (2K steps) | ~1.5 hr T4 |
| Generate samples | ~1 min |

⚠️  Requires T4 GPU (16GB VRAM) — Runtime → Change runtime type → T4

In [ ]:
# @title 1. Clone Repo + Install (~2 min)
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm
!pip install -q transformers accelerate peft bitsandbytes
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    assert gpu_mem >= 14, "⚠️  Need T4/L4 GPU (16GB). Runtime → Change runtime type"


In [ ]:
# @title 2. Preprocess + Train Tokenizer (~3 min)
# Generates training pairs + BPE tokenizer (same as PoetryDuelGPT pipeline)
# Qwen uses its own tokenizer + our 335 special tokens, not the BPE model.
# But we still need poetry_corpus.txt for training data.
%cd /content/poetry-dual-gpt

# Standard:
!python src/preprocess.py

# Curriculum (optional):
# !python src/preprocess.py --curriculum

# Train BPE tokenizer — needed for special token list
!python src/train_bpe.py

print("\n✅ Corpus ready. Special tokens auto-collected from corpus.")


In [ ]:
# @title 🔍 Verify: Qwen loads correctly
%cd /content/poetry-dual-gpt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Test loading (without training — just verify memory fits)
print("Loading Qwen2.5-1.5B in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Test special token addition
import sys; sys.path.insert(0, '/content/poetry-dual-gpt/src')
from train_bpe import build_special_tokens
special_tokens = build_special_tokens("data/poetry_corpus.txt")
num_added = tokenizer.add_tokens(special_tokens, special_tokens=True)
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added} special tokens → vocab: {len(tokenizer):,}")

# Test tokenization
test_line = "<|start|> [LUC_BAT] [RHYME:ong] [TONE:BBBTTB] thân em như chẽn lúa đòng <|reply|>"
ids = tokenizer.encode(test_line, add_special_tokens=False)
print(f"Test line: {len(ids)} tokens")
for t in ['[RHYME:ong]', '[TONE:BBBTTB]', '[LUC_BAT]', '<|start|>', '<|reply|>']:
    tid = tokenizer.convert_tokens_to_ids(t)
    print(f"  {t:20s} → id={tid}")

del model  # free memory for training
torch.cuda.empty_cache()
print("\n✅ Qwen loads with special tokens. Ready for training.")


In [ ]:
# @title 3. Stage 1 — Pretrain on ALL genres (~3 hr T4)
# 5K steps with QLoRA on full poetry corpus
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"

%cd /content/poetry-dual-gpt

# Standard:
!python finetune/finetune.py --stage 1

# Curriculum (if you used --curriculum in preprocess):
# !python finetune/finetune.py --stage 1 --corpus data/poetry_corpus_curriculum.txt

print('\n✅ Stage 1 → checkpoints/qwen_stage1_best/')


In [ ]:
# @title 🔍 Verify: Stage 1 (quick generation test)
%cd /content/poetry-dual-gpt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import os

ckpt = 'checkpoints/qwen_stage1_best'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/qwen_stage1_final'

if os.path.exists(ckpt):
    print(f'Loading {ckpt}...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B", quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(ckpt, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, ckpt)
    model.eval()

    prompt = "<|start|> [LUC_BAT] [RHYME:ong] [TONE:BBBTTB] thân em như chẽn lúa đòng <|reply|>"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model.generate(**inputs, max_new_tokens=20, temperature=0.75,
                                  top_p=0.92, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract reply
    if '<|reply|>' in text:
        reply = text.split('<|reply|>')[1].split('<|end|>')[0].strip() if '<|end|>' in text.split('<|reply|>')[1] else text.split('<|reply|>')[1][:100]
    else:
        reply = text[-100:]
    print(f'Prompt: thân em như chẽn lúa đòng')
    print(f'Reply:  {reply}')
else:
    print('❌ No checkpoint found!')


In [ ]:
# @title 4. Prepare Lục Bát-only corpus (~30 sec)
%cd /content/poetry-dual-gpt

with open('data/poetry_corpus.txt') as f:
    lines = f.readlines()
luc_bat = [l for l in lines if '[LUC_BAT]' in l]
with open('data/corpus_luc_bat.txt', 'w') as f:
    f.writelines(luc_bat)
print(f'Lục Bát pairs: {len(luc_bat):,} / {len(lines):,} total')


In [ ]:
# @title 5. Stage 2 — Fine-tune Lục Bát (~1.5 hr T4)
assert torch.cuda.is_available(), "⚠️  Enable GPU"

%cd /content/poetry-dual-gpt

!python finetune/finetune.py --stage 2 --resume checkpoints/qwen_stage1_best

print('\n🎉 Qwen QLoRA training complete!')
print('   Stage 1: checkpoints/qwen_stage1_best/')
print('   Stage 2: checkpoints/qwen_stage2_best/')


In [ ]:
# @title 🔍 Verify: Stage 2 (compare both stages)
%cd /content/poetry-dual-gpt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import os

prompts = [
    ("Lục Bát", "<|start|> [LUC_BAT] [RHYME:ong] [TONE:BBBTTB] thân em như chẽn lúa đòng <|reply|>"),
    ("Lục Bát", "<|start|> [LUC_BAT] [RHYME:au] [TONE:BBBBTT] rủ nhau xuống biển mò cua <|reply|>"),
    ("Thất Ngôn", "<|start|> [THAT_NGON] [LINK2:B] [DOIAM:TTBBTTB] lom khom dưới núi tiều vài chú <|reply|>"),
]

for stage_label, ckpt in [("Stage 1", "checkpoints/qwen_stage1_best"),
                           ("Stage 2", "checkpoints/qwen_stage2_best")]:
    if not os.path.exists(ckpt):
        if 'final' in ckpt: continue
        ckpt = ckpt.replace('_best', '_final')
    if not os.path.exists(ckpt):
        print(f'❌ {ckpt} not found')
        continue

    print(f'\n{"="*60}')
    print(f'{stage_label} ({ckpt})')
    print(f'{"="*60}')

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                     bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B",
        quantization_config=bnb_config, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(ckpt, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, ckpt)
    model.eval()

    for genre, prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model.generate(**inputs, max_new_tokens=20, temperature=0.75,
                                      top_p=0.92, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(outputs[0], skip_special_tokens=False)
        if '<|reply|>' in text:
            reply = text.split('<|reply|>')[1]
            if '<|end|>' in reply: reply = reply.split('<|end|>')[0]
            reply = reply.strip()[:100]
        else:
            reply = text[-80:]
        prompt_text = prompt.split('<|reply|>')[0].replace('<|start|> ', '').strip()
        print(f'  [{genre}] Prompt: {prompt_text[-60:]}')
        print(f'          Reply:  {reply}')

    del base, model
    torch.cuda.empty_cache()

print('\n✅ Done! Compare: Stage 2 should be more fluent on Lục Bát.')


In [ ]:
# @title 💾 Download LoRA Weights
from google.colab import files
import zipfile, os

for stage_dir in ['checkpoints/qwen_stage1_best', 'checkpoints/qwen_stage2_best',
                   'checkpoints/qwen_stage1_final', 'checkpoints/qwen_stage2_final']:
    if os.path.exists(stage_dir):
        zip_name = f'{stage_dir.replace("/", "_")}.zip'
        with zipfile.ZipFile(zip_name, 'w') as z:
            for root, dirs, files in os.walk(stage_dir):
                for f in files:
                    z.write(os.path.join(root, f))
        print(f'Zipped: {zip_name}')
        files.download(zip_name)

print('\n⬆️  Place in checkpoints/ in your local repo for evaluation.')
